Task 3.2 — Retry with exponential backoff
Apply .with_retry(stop_after_attempt=3, wait_exponential_jitter=True) to a chain that randomly raises a rate-limit-style exception. Add a RunnableLambda before the retry wrapper that logs attempt number using a closure. Verify via logs that retries actually fire and that they don't retry on ValueError (only on your custom RateLimitError).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda, RunnableSequence
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.exceptions import ValueError
from Exception import Exception
import random
import time
from typing import Any

In [ ]:
# 1. Create a chain, chat model will have .with_retry() option, do I need to use .with_fallbacks() option also?
# 2. Inside the chain before chat model call, insert runnable lambda that raises a RateLimitError randomly
# 3. Add a RunnableLambda inside the chain that logs each attempt with a number using closure.
# 4. Verify in logs that retry happens only in case of RateLimitError and not in case of ValueError 

In [ ]:
def log_attempt_closure(x):
    attempt_no = [0]
    logs = []

    def log_attempt():
        attempt_no[0] += 1
        logs.append([{'time': time.now(), 'attempt_no': attempt_no, 'input_prompt_details': x}])
        return x
    return log_attempt

In [ ]:
class RateLimitError(Exception):
    def raise_error():
        if random.random()<0.5:
            raise ValueError
        raise RateLimitError
    
logs={}
def log_entries(x: Any, no_attempts: int) -> None:
    logs.add['session_id'] = {x, no_attempts}


query = 'you have a {role} and years of experience as {yoe} do this particula {task}.'
prompt_template = ChatPromptTemplate.from_messages(query)

chat_model = ChatGoogleGenerativeAI(model='', api_key='')

str_parser = StrOutputParser()

no_attempts = 0

sequential_chain = RunnableSequence(
    RunnableLambda(log_attempt_closure),
    prompt_template, 
    chat_model,
    str_parser).with_retry(
        stop_after_attempt=3,           # Stop after 3 attempts
        wait_exponential_jitter=True    # Introduce jitter, so as to overcome the thundering herd problem
        retry_if_exception_type=(RateLimitError,)
    )

sequential_chain.invoke({'role': role, 'yoe': yoe, 'task': task})

# a simple solution would be to introduce a new method and wrap it with runnable lambda.
# that function in turn will be a wrapper for chat_model and implements invoke within it. 
    # in case error happens log it first with model_name, input_prompt details and return the same error back 